# 008 — Grokking on modular addition

Runner for `scripts/008_grokking.py` on a Colab GPU.

**The implementation is not in this notebook.** It lives in `src/slt/grokking.py` and
`scripts/008_grokking.py`, and this file only drives them (CONSTITUTION 4e). If you find
yourself editing model code in a cell, edit the repo and re-run the clone cell instead —
a notebook that quietly forks the model is how a run stops being reproducible.

**What it runs.** A 1-layer transformer ($d_\text{model}=128$, 4 heads, $d_\text{mlp}=512$,
no LayerNorm, 226,816 parameters) on $(a+b) \bmod 113$, trained full-batch with AdamW
(lr $10^{-3}$, weight decay 1.0) on 30% of the 12,769 pairs for 40,000 steps. Train
accuracy saturates within ~1k steps; test accuracy sits at chance for roughly another
decade of steps and then climbs to ~100%. That gap is the plot.

**Expected wall clock:** ~10–25 min on a T4, less on an L4/A100. This is an estimate, not
a measurement (CONSTITUTION 9e) — the training cell prints a live step/s rate and ETA, so
record the real number and put it in the log entry.

**Cell order:** check GPU → clone repo → (optional) put `runs/` on Drive → 200-step smoke
test → start TensorBoard → the real run → plot → sweeps over lr / optimizer / depth →
numbers for the log → download.

Set the runtime first: **Runtime ▸ Change runtime type ▸ GPU**.

---
## Before you run: the hypothesis

CONSTITUTION 1. This entry is an EXPERIMENT, and the question goes on record *before* the
run, not after the curves are on screen. Fill this in and paste it into the
`## Hypothesis` section of `logs/008-grokking.md`.

> **Hypothesis:**
>
> **If true, we expect:**
>
> **If false, we expect:**

"No hypothesis, I'm just looking" is a valid answer — write *that* down and the entry is
recorded as exploratory. What it cannot be is blank.

One thing the entry already commits to (rule 0a): this run measures no SLT quantity, so
there is nothing here for a cheap baseline to beat, and 008 must not be cited later as
evidence about SLT. The $\|w\|_2$ curve it logs is the baseline that a *later* SLT claim
would have to beat.

---
## 1. Check the GPU

In [ ]:
import subprocess

try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu = "none visible — set Runtime > Change runtime type > GPU, or expect the CPU path (~1.5-3 h)"
print("gpu    ", gpu)

import torch
print("torch  ", torch.__version__)
print("cuda   ", torch.cuda.is_available())

---
## 2. Get the repo

Public, so no credentials are needed. If it is ever made private, put a token in
**Colab ▸ 🔑 Secrets** and read it with `google.colab.userdata.get(...)` — never paste a
token into a cell, because the cell output gets saved into the notebook.

In [ ]:
import pathlib

REPO_URL = "https://github.com/alexali04/slt.git"
REPO_DIR = pathlib.Path("/content/slt")

if REPO_DIR.exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

---
## 3. Optional: keep `runs/` on Drive

Recommended for the full run. Colab disconnects, and `runs/008-grokking/` is where the
metrics and checkpoints go (CONSTITUTION 9a). On Drive, a disconnect costs you the GPU
and not the run: reconnect, re-run cells 1–3, and add `--resume` to the training command.

Leave `USE_DRIVE = False` for a quick throwaway run.

In [ ]:
USE_DRIVE = True
DRIVE_RUNS = "/content/drive/MyDrive/slt-runs"

runs = REPO_DIR / "runs"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    target = pathlib.Path(DRIVE_RUNS)
    target.mkdir(parents=True, exist_ok=True)

    if runs.is_symlink():
        pass                                    # already pointed somewhere
    elif runs.exists() and any(runs.iterdir()):
        print(f"{runs} already holds data — leaving it where it is")
    else:
        if runs.exists():
            runs.rmdir()                        # empty, safe to replace
        runs.symlink_to(target)

print("runs ->", runs.resolve() if runs.exists() else "(created on first run)")

---
## 4. Smoke test — 200 steps

Proves the pipeline end to end (train writes a run directory, plot reads it) in well under
a minute, before you commit half an hour of GPU to it. `--tag smoke` keeps it in its own
run directory and gives its figures their own filenames, so it cannot overwrite the real
ones.

There is nothing to see in these curves. 200 steps is before anything happens.

In [ ]:
!python -u scripts/008_grokking.py train --steps 200 --tag smoke --print-every 50 --ckpt-every 0 --tensorboard
!python -u scripts/008_grokking.py plot --tag smoke

---
## 5. Live tracking — start TensorBoard first

Run this **before** the training cell and leave it open: train/test accuracy, train/test
loss, $\|w\|_2$ and the learning rate all stream in as the run goes, and the dashboard
refreshes itself. It stays useful across the sweep cells too — `--logdir runs` picks up
every run directory, so variants land on the same axes as they finish.

Pass `--tensorboard` to any `train` command to feed it. Without that flag nothing is
streamed and the run is otherwise identical.

This does **not** replace the figures. `metrics.csv` remains the record and
`plot` still produces the raw and diagram PNGs from it (CONSTITUTION 9a); TensorBoard is
for watching, not for reporting. Nothing gets read out of here into the log entry that
is not also in the CSV.

Colab has `tensorboard` preinstalled. Elsewhere: `pip install tensorboard`.

*If the panels stay empty:* nothing has been written yet — start the training cell, wait
for the first progress line, then hit refresh (↻) top-right. If `runs/` is on Drive
(cell 3), reads go through the Drive mount and updates lag by a few seconds.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

---
## 6. The real run — 40,000 steps

Progress prints every 500 steps with a live rate and ETA. Interrupting the cell is safe:
the script catches it, writes `last.pt`, and the metrics written so far are already
flushed to disk, so a partial run still plots (CONSTITUTION 9b).

To continue an interrupted or disconnected run, add `--resume`.

In [ ]:
!python -u scripts/008_grokking.py train --device auto --tensorboard

In [ ]:
# Continue where a stopped run left off. No-op if there is no checkpoint.
# !python -u scripts/008_grokking.py train --device auto --tensorboard --resume

---
## 7. The figures

`figures/raw/008_grokking_curves.png` is the object with nothing written on it
(CONSTITUTION 8a) — look at that one first, before the annotated version tells you what to
see. `figures/diagram/008_grokking.png` adds the measured transition points (8b).

**`TAG` selects which run to plot.** Leave it `""` for the reference configuration. After
a sweep, set it to the tag the training cell printed — `--n-layers 2` writes the run
under tag `L2`, so plotting with `TAG = ""` would silently draw the *reference* run
instead and label it "1 block", which is accurate and not what you asked for. Run the
`list` cell below to see every tag you have.

In [ ]:
from IPython.display import Image, display

TAG = ""          # "" = the reference run; e.g. "L2", "lr0.003", "L2-lr0.003"

flag = f"--tag {TAG}" if TAG else ""
suffix = f"_{TAG}" if TAG else ""

!python -u scripts/008_grokking.py plot {flag}

display(Image(f"figures/raw/008_grokking_curves{suffix}.png"))
display(Image(f"figures/diagram/008_grokking{suffix}.png"))

---
## 8. Sweeps — what if I change X?

`train --help` lists every flag. Anything left alone is the reference configuration;
anything you change renames the run directory and the figures after the change, so a
sweep cannot overwrite itself. `--lr 3e-3 --n-layers 2` writes `runs/008-grokking-L2-lr0.003/`.

Each variant is another full run, so budget the wall clock accordingly — five variants is
five times the cost. Drop `--steps` to 15,000 for a coarse sweep if you only care about
whether the transition happens and roughly when.

**These runs are not entry 008.** 008 is the replication at the reference config. A sweep
is a new question, so it is a new log entry with its own hypothesis (CONSTITUTION 1) —
and "does grokking survive X?" is a real hypothesis, worth writing down before you look.

In [ ]:
# Each line is one variant. Comment out what you do not want; every one is a full run.
SWEEPS = [
    "--n-layers 2",                    # two blocks
    "--lr 3e-3",                       # 3x the learning rate
    "--weight-decay 0.1",              # a tenth the decay
    "--weight-decay 0.0",              # none at all
    "--optimizer adam",                # L2-in-gradient instead of decoupled
    # "--optimizer sgd --lr 0.1",      # no adaptive scaling; needs a much bigger lr
    # "--train-frac 0.5",              # more of the pairs in training
    # "--seed 1",                      # same config, different draw
]

STEPS = 15000   # coarse sweep; raise to 40000 to match the reference run

for flags in SWEEPS:
    print(f"\n{'=' * 70}\n  {flags}\n{'=' * 70}", flush=True)
    !python -u scripts/008_grokking.py train --device auto --tensorboard --steps {STEPS} {flags}

In [ ]:
# Every run so far, with its settings and where it landed.
!python -u scripts/008_grokking.py list

In [ ]:
# Overlay them. Pass --runs with a subset of the names printed above to narrow it down.
!python -u scripts/008_grokking.py compare

display(Image("figures/diagram/008_compare.png"))

---
## 9. Numbers for the log entry

`runs/` and `figures/` are both git-ignored, so nothing here reaches the repo on its own.
Anything that matters gets copied into `logs/008-grokking.md` by hand (CONSTITUTION 9d),
including the actual wall clock against the estimate (9e).

In [ ]:
import json
import numpy as np

run = pathlib.Path("runs/008-grokking")
cfg = json.loads((run / "config.json").read_text())
rows = np.genfromtxt(run / "metrics.csv", delimiter=",", names=True)

steps, secs = rows["step"][-1], rows["elapsed_s"][-1]
print(f"device          {cfg['device']}  ({cfg['n_params']:,} parameters)")
print(f"reached step    {int(steps):,} of {cfg['config']['steps']:,}")
print(f"actual cost     {secs / 60:.1f} min   ({steps / secs:.1f} step/s)")
print(f"torch           {cfg['torch']}")
print()
print("milestones as printed by `plot` above go in the Result section;")
print("this line goes in Method, next to the estimate:")
print(f"  **Actual cost:** {secs / 60:.1f} min on {cfg['device']} ({steps / secs:.1f} step/s)")

---
## 10. Take the results with you

The zip holds the run directory (metrics, config, checkpoints) and both figures. The
checkpoints are the reason to keep it: they are what a local-learning-coefficient estimate
along the trajectory would run on later, without retraining.

In [ ]:
from google.colab import files

!zip -qr /content/008_grokking_results.zip runs/008-grokking figures/raw/008_grokking_curves.png figures/diagram/008_grokking.png
!du -h /content/008_grokking_results.zip

files.download("/content/008_grokking_results.zip")

---
## After the run

1. Paste the hypothesis you wrote at the top into `logs/008-grokking.md`, and mark the
   entry exploratory if that is what it was.
2. Fill in **Result**, **Conclusion**, and the **Actual cost** line from the numbers above.
3. Update the 008 row in `LOG.md` — it currently says *not yet run* — and drop the entry
   from the **Not yet run** list.
4. Commit the log entry. Do not commit `runs/` or `figures/`; both are ignored on purpose.